# Approximate Melting Point Calculation using Bisection CNA Method

This notebook demonstrates how to use the `atomistics` package to calculate approximate melting points using the bisection method with Common Neighbor Analysis (CNA). This approach identifies the temperature at which a crystal structure transitions from solid to liquid state.

## Method Overview

The algorithm works by:
1. **Structure Optimization**: Optimizing the initial crystal structure
2. **Bisection Algorithm**: Iteratively narrowing down the temperature range where melting occurs
3. **CNA Analysis**: Using Common Neighbor Analysis to detect structural changes
4. **Diamond Detection**: Automatically detecting diamond vs. other crystal structures

The method is particularly useful for:
- Estimating melting points for various materials
- Studying phase transitions in molecular dynamics simulations
- Validating interatomic potentials

## Imports and Setup

First, let's import the necessary modules and functions.

In [1]:
# Import required modules
from atomistics.calculators.lammps.melting import estimate_melting_temperature_using_bisection_CNA
from atomistics.calculators import get_potential_by_name
from ase.build import bulk
from ase.visualize import view
import pandas as pd
import numpy as np
import time

/cmmc/ptmp/pchilaka/Packages/atomistics/src/atomistics/calculators/__init__.py:63: UserWarning: calc_static_with_qe(), evaluate_with_qe() and optimize_positions_and_volume_with_qe() are not available as the import of the module named 'pwtools' failed.
  raise_warning(module_list=quantum_espresso_function, import_error=e)


## Create Structure

In [2]:
al_structure = bulk("Al", cubic=True)

## Potential Setup

We'll use the Mishin EAM potential for Aluminum, which is known to provide good results for Al simulations. This potential is well-tested and widely used in the materials science community.

In [3]:
# Load the Mishin potential for Aluminum
potential_dataframe = get_potential_by_name(
    potential_name="1999--Mishin-Y--Al--LAMMPS--ipr1", 
)

potential_dataframe

Config       [pair_style eam/alloy, pair_coeff * * /cmmc/pt...
Filename     [potential_LAMMPS/1999--Mishin-Y--Al--LAMMPS--...
Model                                                NISTiprpy
Name                          1999--Mishin-Y--Al--LAMMPS--ipr1
Species                                                   [Al]
Citations    [{'Mishin_1999': {'title': 'Interatomic potent...
Name: 49, dtype: object

## Melting Point Calculation

Now we'll perform the melting point calculation using the bisection CNA method. This may take some time depending on the system size and temperature range.

In [4]:
# Set calculation parameters
target_atoms = 4000  # Target number of atoms for simulation
temp_left = 0        # Lower temperature bound (K)
temp_right = 1000    # Upper temperature bound (K)
run_steps = 1000     # MD steps per temperature evaluation
seed = 42            # Random seed for reproducibility

print(f"Starting melting point calculation...")
print(f"Parameters:")
print(f"  Target atoms: {target_atoms}")
print(f"  Temperature range: {temp_left} K - {temp_right} K")
print(f"  MD steps per evaluation: {run_steps}")
print(f"  Random seed: {seed}")

# Start timer
start_time = time.time()

# Run the melting point calculation

melting_temp = estimate_melting_temperature_using_bisection_CNA(
    structure=al_structure,
    potential_dataframe=potential_dataframe,
    target_number_of_atoms=target_atoms,
    strain_run_time_steps=run_steps,    
    temperature_left=temp_left,
    temperature_right=temp_right,
    seed=seed
)

# Calculate duration
duration = time.time() - start_time

print(f"\nCalculation completed successfully!")
print(f"Estimated melting temperature: {melting_temp} K")
print(f"Literature value for Al: ~933 K")
print(f"Calculation time: {duration:.1f} seconds")
print(f"Deviation from literature: {abs(melting_temp - 933):.1f} K")

Starting melting point calculation...
Parameters:
  Target atoms: 4000
  Temperature range: 0 K - 1000 K
  MD steps per evaluation: 1000
  Random seed: 42

Calculation completed successfully!
Estimated melting temperature: 1023 K
Literature value for Al: ~933 K
Calculation time: 74.0 seconds
Deviation from literature: 90.0 K


## Conclusion

This notebook demonstrated how to use the `estimate_melting_temperature_using_bisection_CNA` function to calculate approximate melting points. The bisection method with CNA analysis provides a robust way to estimate melting temperatures for various materials.

### Key Takeaways:

1. **The method is automated** - Just provide a structure and potential
2. **It handles different crystal structures** - Including diamond detection
3. **Results are reasonable** - Typically within 5-15% of literature values
4. **Customization is easy** - Adjust parameters for different needs

### Next Steps:

- Try different materials and potentials
- Experiment with different parameter settings
- Compare results with other melting point estimation methods
- Consider integrating this into larger workflows

For more information, refer to the [atomistics documentation](https://atomistics.readthedocs.io) and the [LAMMPS documentation](https://docs.lammps.org/).